In [1]:
import hashlib
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization

class DigitalWallet:
    def __init__(self, owner_name):
        self.owner_name = owner_name
        # STEP 1: Generate Private/Public Key Pair (RSA)
        self.private_key = rsa.generate_private_key(
            public_exponent=65537,
            key_size=2048
        )
        self.public_key = self.private_key.public_key()

        # STEP 2: Derive Wallet Address (SHA-256 hash of Public Key)
        pub_bytes = self.public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        )
        self.address = hashlib.sha256(pub_bytes).hexdigest()[:20]

    def sign_transaction(self, transaction_data):
        """Signs the transaction using the Private Key"""
        signature = self.private_key.sign(
            transaction_data.encode(),
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        return signature

    @staticmethod
    def verify_transaction(public_key, transaction_data, signature):
        """Verifies the signature using the Public Key"""
        try:
            public_key.verify(
                signature,
                transaction_data.encode(),
                padding.PSS(
                    mgf=padding.MGF1(hashes.SHA256()),
                    salt_length=padding.PSS.MAX_LENGTH
                ),
                hashes.SHA256()
            )
            return True
        except Exception:
            return False

# --- SIMULATION STEPS ---

# 1. Initialize Wallets
alice_wallet = DigitalWallet("Alice")
bob_wallet = DigitalWallet("Bob")

print(f"Alice's Address: {alice_wallet.address}")
print(f"Bob's Address: {bob_wallet.address}")

# 2. Create a Transaction
tx_data = f"Sender: {alice_wallet.address}, Receiver: {bob_wallet.address}, Amount: 50 BTC"

# 3. Alice Signs the Transaction
print("\n[Step 1] Alice is signing the transaction...")
signature = alice_wallet.sign_transaction(tx_data)

# 4. Verification Tests
print("\n[Step 2] Verifying legitimate transaction...")
is_valid = DigitalWallet.verify_transaction(alice_wallet.public_key, tx_data, signature)
if is_valid:
    print("RESULT: Transaction Validated Successfully!")
else:
    print("RESULT: Invalid Signature!")

# 5. Tampering Test
print("\n[Step 3] Simulation: A hacker tries to change the amount to 5000 BTC...")
tampered_tx_data = f"Sender: {alice_wallet.address}, Receiver: {bob_wallet.address}, Amount: 5000 BTC"
is_tampered_valid = DigitalWallet.verify_transaction(alice_wallet.public_key, tampered_tx_data, signature)

if not is_tampered_valid:
    print("RESULT: Verification Failed! Tampering Detected.")

Alice's Address: 38eb8bab20339d947116
Bob's Address: fe3dc086560d85e402bc

[Step 1] Alice is signing the transaction...

[Step 2] Verifying legitimate transaction...
RESULT: Transaction Validated Successfully!

[Step 3] Simulation: A hacker tries to change the amount to 5000 BTC...
RESULT: Verification Failed! Tampering Detected.
